# Verification: Dembo & Goldstein (1978) Equilibrium Theory with Ring Closure

Compares BNGL parameter_scan (ODE with `max_stoich={R=>6}`) against
Dembo & Goldstein (1978) analytical equilibrium theory (Eq. 14), which
includes chain-length-dependent ring closure: $J_i = 4J_2/i^2$ for $i \ge 2$.

Ring closure is implemented as bidirectional rules for each chain size
$n = 2, \ldots, 6$, with forward rate $j^+_n = 2n \cdot J_n \cdot k_\text{off}$
to compensate for the BNG reverse multiplicity of $2n \cdot k_\text{off}$.

- **Left panel**: BNGL chain-only (k=5) vs analytical $J_2=0$
- **Right panel**: BNGL $J_2=0$ and $J_2=10$ (k=10) vs Dembo Fig. 3 analytical curves

In [ ]:
import subprocess, os, glob, shutil
import numpy as np
from scipy.optimize import brentq
import matplotlib.pyplot as plt

os.chdir(os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else '.')

# Physical parameters (must match BNGL file)
NA = 6.02214076e23
V_cell = 1e-9  # L/cell
R_per_cell = 3e5
RT = R_per_cell
Ka = 1e6 / 0.01  # kon/koff = 1e8 /M
KxRT = 5  # k in Dembo notation

def to_a(conc_molecules):
    """Convert molecule count to Dembo nondimensional a = H*A."""
    return 2 * Ka * conc_molecules / (NA * V_cell)

def solve_dembo_eq14(a, k, J1=0, J2=0, b=0):
    """Solve Dembo & Goldstein (1978) Eq. 14 for x (free antibody fraction).
    Returns (x, w1, x_poly)."""
    if a <= 0:
        return 1.0, 1.0, 0.0
    x_upper = min(1.0, 0.999999 / (2 * a * k))
    if x_upper <= 0:
        return 0.0, 0.0, 0.0
    def f(x):
        s = 2 * a * k * x
        val = (1 + a + b/2)**2 * x / (1 - s)**2
        if J2 > 0:
            val -= (4 * J2 / k) * np.log(1 - s)
        val += 2 * a * x * (J1 - 4 * J2)
        return val - 1
    try:
        x = brentq(f, 0, x_upper, xtol=1e-14)
    except ValueError:
        return 0.0, 0.0, 0.0
    w1 = ((1 + a + b/2)**2 + 2 * a * J1) * x
    return x, w1, max(0, 1 - w1)

def dembo_xpoly(a_arr, k, J1=0, J2=0, b=0):
    """Compute x_poly for array of a values."""
    return np.array([solve_dembo_eq14(a, k, J1, J2, b)[2] for a in a_arr])

print('Dembo solver ready')

In [ ]:
# Run BNGL parameter scans (k=5, k=10, k=10+J2=10)
bngl_file = 'symmetric_bivalent_hapten_receptor_crosslinking_dembo1978_with_rings.bngl'
prefix = 'symmetric_bivalent_hapten_receptor_crosslinking_dembo1978_with_rings'
ref_dir = 'reference'

scan_names = ['scan_k5', 'scan_k10', 'scan_k10_J2_10']
ref_paths = {s: os.path.join(ref_dir, f'{prefix}_{s}.scan') for s in scan_names}

# Use reference data if available; otherwise run BNG
if all(os.path.exists(p) for p in ref_paths.values()):
    print('Using reference data from reference/')
else:
    # Clean transient artifacts
    for f in glob.glob(prefix + '*'):
        if f.endswith(('.bngl', '.ipynb', '.png')):
            continue
        if os.path.isdir(f):
            shutil.rmtree(f)
        else:
            os.remove(f)

    print(f'Running {bngl_file}...', flush=True)
    result = subprocess.run(['bionetgen', 'run', '-i', bngl_file],
                            capture_output=True, text=True, timeout=120)
    print(f'RC={result.returncode}')
    if result.returncode != 0:
        print(result.stderr[:500])

    # Copy scan files to reference/, clean transient files
    os.makedirs(ref_dir, exist_ok=True)
    for s in scan_names:
        src = f'{prefix}_{s}.scan'
        if os.path.exists(src):
            shutil.copy2(src, ref_paths[s])
    for f in glob.glob(prefix + '*'):
        if f.endswith(('.bngl', '.ipynb', '.png')):
            continue
        if os.path.isdir(f):
            shutil.rmtree(f)
        else:
            os.remove(f)

def parse_scan(scan_path):
    """Parse scan file -> (a_free, xpoly_sim)."""
    d = np.loadtxt(scan_path, comments='#')
    a_free = to_a(d[:, 1])
    xpoly = 1 - (d[:, 2] + d[:, 3] + d[:, 4]) / RT
    return a_free, xpoly

a_free_k5, xpoly_k5 = parse_scan(ref_paths['scan_k5'])
a_free_k10, xpoly_k10 = parse_scan(ref_paths['scan_k10'])
a_free_k10j, xpoly_k10j = parse_scan(ref_paths['scan_k10_J2_10'])

for lab, af, xp in [('k=5,J2=0', a_free_k5, xpoly_k5),
                     ('k=10,J2=0', a_free_k10, xpoly_k10),
                     ('k=10,J2=10', a_free_k10j, xpoly_k10j)]:
    pk = np.argmax(xp)
    print(f'{lab}: peak x_poly={xp[pk]:.4f} at a_free={af[pk]:.3f}')

In [ ]:
# Analytical solutions
a_fine = np.logspace(-3, 3, 300)

# Chain-only (J2=0) for validation against BNGL
xpoly_J0 = dembo_xpoly(a_fine, k=KxRT, J2=0)

# Effect of J2 (Dembo Fig. 3: k=10, J1=0, b=0)
k_fig3 = 10
J2_values = [0, 10, 100, 1000]
xpoly_fig3 = {j2: dembo_xpoly(a_fine, k=k_fig3, J2=j2) for j2 in J2_values}

print('Analytical solutions computed')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: BNGL validation (k=5, chain-only)
ax1.plot(a_fine, xpoly_J0, 'k-', lw=2, label=f'Analytical ($J_2$=0, k={KxRT})')
ax1.plot(a_free_k5, xpoly_k5, 'ro', ms=4, alpha=0.7, label='BNGL chain-only')
ax1.set_xscale('log')
ax1.set_xlabel('$a = H \cdot A_{\mathrm{free}}$ (nondimensional)')
ax1.set_ylabel('$x_{\mathrm{poly}}$ (fraction cross-linked)')
ax1.set_title(f'Chain-only validation (k={KxRT})')
ax1.legend(fontsize=9)
ax1.set_xlim(1e-3, 1e3)
ax1.set_ylim(0, 1)

# Right: Dembo Fig. 3 (k=10) with BNGL overlays
colors = {0: 'C0', 10: 'C1', 100: 'C2', 1000: 'C3'}
for j2 in J2_values:
    ax2.plot(a_fine, xpoly_fig3[j2], '-', color=colors[j2], lw=2,
             label=f'Analytical $J_2$={j2}')
ax2.plot(a_free_k10, xpoly_k10, 'ko', ms=4, alpha=0.5,
         label='BNGL $J_2$=0')
ax2.plot(a_free_k10j, xpoly_k10j, 's', color='C1', ms=5, alpha=0.7,
         mfc='none', mew=1.5, label='BNGL $J_2$=10')
ax2.set_xscale('log')
ax2.set_xlabel('$a = H \cdot A_{\mathrm{free}}$ (nondimensional)')
ax2.set_ylabel('$x_{\mathrm{poly}}$ (fraction cross-linked)')
ax2.set_title(f'Ring closure effect (k={k_fig3}, $J_1$=0)')
ax2.legend(fontsize=8)
ax2.set_xlim(1e-3, 1e3)
ax2.set_ylim(0, 1)

fig.suptitle('Dembo & Goldstein (1978): Equilibrium cross-linking with rings',
             fontsize=12)
fig.tight_layout()
fig.savefig('verify_dembo1978_with_rings.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved verify_dembo1978_with_rings.png')

In [ ]:
# Quantitative validation: BNGL vs analytical (using a_free)
max_err = 0
for label, a_f, xp_s, k_val, j2 in [('k=5,  J2=0 ', a_free_k5, xpoly_k5, KxRT, 0),
                                       ('k=10, J2=0 ', a_free_k10, xpoly_k10, k_fig3, 0),
                                       ('k=10, J2=10', a_free_k10j, xpoly_k10j, k_fig3, 10)]:
    xp_ana = dembo_xpoly(a_f, k=k_val, J2=j2)
    err = np.abs(xp_s - xp_ana)
    mask = xp_ana > 0.01
    rel = (err[mask] / xp_ana[mask]).max() if mask.any() else 0
    pk = np.argmax(xp_s)
    max_err = max(max_err, err.max())
    print(f'{label}: max_abs_err={err.max():.5f}  max_rel_err={rel:.4f}  '
          f'peak={xp_s[pk]:.4f} at a={a_f[pk]:.3f}')

print()
if max_err < 0.02:
    print('PASS: BNGL matches Dembo Eq. 14 for all scans.')
else:
    print('FAIL: Significant disagreement detected.')